# **Comparativa de algoritmos de clustering: K-Means, Jerárquico, DBSCAN, GNM**

## El dataset común será *"marketing_campaing.csv"*, disponible en Kaggle, el cuál dispone de información demográfica, comportamiento de compra y respuestas a campañas de marketing.

## El objetivo de los clusters será organizar grupos distintivos con información de negocio relevante para personalizar futuras campañas y optimizar la asignación de recursos promocionales.

In [ ]:
# =============================================================================
# CELDA 0 · Verificación e instalación de dependencias
# =============================================================================
import subprocess, sys

_required = [
    "numpy", "pandas", "matplotlib", "seaborn",
    "scikit-learn", "scipy", "jinja2"   # jinja2 necesario para df.style
]

for pkg in _required:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

print("✓ Todas las dependencias están disponibles.")

In [ ]:
# =============================================================================
# CELDA 1 · Configuración centralizada
# Todos los parámetros configurables del proyecto se definen aquí.
# Modificar este bloque es suficiente para adaptar el experimento.
# =============================================================================

# --- Rutas ---
DATA_PATH = "C:\\Users\\Nimo\\Documents\\Clases\\EUSA\\Sistemas de Aprendizaje Automático\\Ejercicios\\Machine Learning No-Supervisado\\marketing_campaign.csv"   # ruta relativa al dataset
DATA_SEP  = ";"                              # el CSV usa punto y coma como separador

# --- Reproducibilidad ---
RANDOM_STATE = 42   # semilla global para todos los algoritmos estocásticos

# --- PCA ---
PCA_N_MIN      = 5          # mínimo de componentes a explorar
PCA_N_MAX      = 10         # máximo de componentes a explorar
PCA_VAR_TARGET = 0.85       # varianza acumulada mínima exigida

# --- K-Means / Jerárquico / GMM ---
K_RANGE = range(2, 11)      # valores de k a explorar (2..10 inclusive)

# --- DBSCAN ---
DBSCAN_MIN_SAMPLES = [3, 5, 8, 10]
DBSCAN_EPS_RANGE   = [round(x, 2) for x in
                      [0.3 + i * (2.5 - 0.3) / 9 for i in range(10)]]
# → genera 10 valores equidistantes entre 0.30 y 2.50

In [ ]:
# =============================================================================
# CELDA 2 · Imports y sistema de diseño visual
# Se define una única vez la paleta, el estilo y los tamaños de figura.
# Reutilizar estas constantes garantiza coherencia visual en todo el notebook.
# =============================================================================

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns


from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
from sklearn.cluster         import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture         import GaussianMixture
from sklearn.metrics         import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

# --- Estilo global ---
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi"      : 120,
    "axes.titlesize"  : 13,
    "axes.labelsize"  : 11,
    "legend.fontsize" : 10,
    "figure.figsize"  : (10, 5),
})

# --- Paleta discreta para clusters (hasta 12 grupos) ---
CLUSTER_PALETTE = sns.color_palette("tab10", 12)

# --- Color especial para ruido (DBSCAN) ---
NOISE_COLOR = "#cccccc"

print("✓ Imports y configuración visual cargados.")

In [ ]:
# =============================================================================
# CELDA 3 · Carga del dataset
# Se usa pd.read_csv con el separador configurado en CELDA 1.
# La variable 'df' es el dataframe original; no se modifica nunca directamente.
# =============================================================================

df = pd.read_csv(DATA_PATH, sep=DATA_SEP)

print(f"Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print("-" * 60)
display(df.head())

In [ ]:
# =============================================================================
# CELDA 4 · Exploración inicial
# =============================================================================

print("=== Tipos de datos y valores no nulos ===")
print(df.info())

print("\n=== Valores faltantes por columna (solo las que los tienen) ===")
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
print(nulls.to_string() if len(nulls) else "No hay valores faltantes.")

print("\n=== Estadísticos descriptivos (variables numéricas) ===")
display(df.describe().T.style.format("{:.2f}"))

In [ ]:
# =============================================================================
# CELDA 5 · Tratamiento de valores faltantes → df_0
# Se trabaja sobre una copia para preservar df original intacto.
# =============================================================================

df_0 = df.copy()

# --- Imputación de Income por mediana ---
# La mediana es robusta frente a outliers, a diferencia de la media.
# Dado que Income muestra distribución asimétrica a la derecha (clientes de
# muy alto poder adquisitivo), la mediana representa mejor el valor central.
income_median = df_0["Income"].median()
n_imputed     = df_0["Income"].isnull().sum()

df_0["Income"] = df_0["Income"].fillna(income_median)

print(f"Columna 'Income': {n_imputed} valor(es) imputado(s) con mediana = {income_median:,.2f} €")

# --- Eliminación de filas con nulos residuales (si los hubiera) ---
n_before = len(df_0)
df_0.dropna(inplace=True)
n_after  = len(df_0)
if n_before != n_after:
    print(f"Eliminadas {n_before - n_after} filas con nulos residuales.")
else:
    print("No hay nulos residuales tras la imputación.")

print(f"\nShape de df_0: {df_0.shape}")

In [ ]:
# =============================================================================
# CELDA 6 · Ingeniería de variables y selección para clustering
# =============================================================================
import datetime

# --- Feature engineering: Age a partir de Year_Birth ---
# La edad es más interpretable que el año de nacimiento y más estable
# como feature de distancia. Se calcula respecto al año de referencia del dataset.
YEAR_REF = 2014   # año aproximado de recogida de datos (según Dt_Customer)
df_0["Age"] = YEAR_REF - df_0["Year_Birth"]

# Verificación de rango: descartar edades implausibles (>90 años)
edad_max = df_0["Age"].max()
edad_min = df_0["Age"].min()
print(f"Rango de Age: {edad_min} – {edad_max} años")
outliers_edad = (df_0["Age"] > 90).sum()
if outliers_edad > 0:
    print(f"   {outliers_edad} registro(s) con Age > 90 → se imputan con mediana")
    df_0["Age"] = df_0["Age"].where(df_0["Age"] <= 90, df_0["Age"].median())

# --- Variables descartadas y motivo ---
# ID              : identificador único, sin valor informativo
# Year_Birth      : sustituida por Age (feature derivada más interpretable)
# Dt_Customer     : fecha en formato string; requeriría parsing adicional
#                   y su señal ya está parcialmente capturada por Recency
# AcceptedCmp1-5  : outcomes de campañas pasadas, no descriptores de comportamiento base
# Response        : etiqueta de la última campaña (outcome)
# Complain        : evento binario puntual, señal muy baja
# Z_CostContact   : constante (valor 3 en todo el dataset)
# Z_Revenue       : constante (valor 11 en todo el dataset)

COLS_NUMERICAS = [
    "Age",
    "Income",
    "Recency",
    "MntWines", "MntFruits", "MntMeatProducts",
    "MntFishProducts", "MntSweetProducts", "MntGoldProds",
    "NumDealsPurchases", "NumWebPurchases",
    "NumCatalogPurchases", "NumStorePurchases",
    "NumWebVisitsMonth",
    "Kidhome", "Teenhome",
]

COLS_CATEGORICAS = ["Education", "Marital_Status"]

print(f"\nVariables numéricas seleccionadas ({len(COLS_NUMERICAS)}):")
for c in COLS_NUMERICAS:
    print(f"  · {c}")
print(f"\nVariables categóricas candidatas: {COLS_CATEGORICAS}")
print(f"\nShape del subconjunto numérico: {df_0[COLS_NUMERICAS].shape}")

In [ ]:
# =============================================================================
# CELDA 7 · Escalado de variables numéricas → X_scaled
# StandardScaler centra cada variable en 0 y la escala a varianza unitaria.
# Es preferible a MinMaxScaler cuando hay outliers (outliers en Income, Mnt*)
# porque MinMaxScaler los comprimiría todos los demás valores.
# Se guarda el objeto scaler para poder aplicar inverse_transform en Tarea 2.
# =============================================================================

scaler  = StandardScaler()
X_num   = df_0[COLS_NUMERICAS].values          # array numpy, sin columnas extra aún

X_scaled = scaler.fit_transform(X_num)         # shape: (n_clientes, 15)

# Verificación rápida: media ≈ 0, std ≈ 1
means = X_scaled.mean(axis=0).round(6)
stds  = X_scaled.std(axis=0).round(4)
check = pd.DataFrame({"variable": COLS_NUMERICAS,
                       "media_escalada": means,
                       "std_escalada"  : stds})

print("Verificación de escalado (deben ser ≈0 y ≈1):")
display(check.style.format({"media_escalada": "{:.6f}", "std_escalada": "{:.4f}"}))
print(f"\nShape de X_scaled: {X_scaled.shape}")

In [ ]:
# =============================================================================
# CELDA 8 · Limpieza de Marital_Status, codificación de categóricas e integración
# =============================================================================

df_1 = df_0.copy()

# --- Inspección previa de categorías reales en el dataset ---
print("Categorías originales en Marital_Status:")
print(df_1["Marital_Status"].value_counts().to_string())

# --- Unificación de categorías minoritarias / erróneas ---
# 'Alone'  → equivalente semántico de 'Single' (vive solo)
# 'YOLO'   → valor no estándar sin significado demográfico claro → 'Single'
# 'Absurd' → valor basura → 'Single' (categoría residual más neutral)
# Esta decisión reduce ruido en el espacio de features sin perder registros.
marital_map = {
    "Married"  : "Married",
    "Together" : "Together",
    "Single"   : "Single",
    "Divorced" : "Divorced",
    "Widow"    : "Widow",
    "Alone"    : "Single",    # equivalente semántico
    "YOLO"     : "Single",    # valor no estándar → categoría residual
    "Absurd"   : "Single",    # valor basura → categoría residual
}
df_1["Marital_Status_clean"] = df_1["Marital_Status"].map(marital_map)

print("\nCategorías tras limpieza:")
print(df_1["Marital_Status_clean"].value_counts().to_string())

# --- Codificación ordinal de Education ---
edu_order = {"Basic": 0, "2n Cycle": 1, "Graduation": 2, "Master": 3, "PhD": 4}
df_1["Education_enc"] = df_1["Education"].map(edu_order)

# Verificar que no quedan NaN (categorías no mapeadas)
assert df_1["Education_enc"].isnull().sum() == 0, \
    "Hay categorías de Education no mapeadas"

# --- One-Hot Encoding de Marital_Status limpio ---
# drop_first=True elimina una categoría de referencia (Divorced)
# para evitar multicolinealidad perfecta entre dummies.
marital_dummies = pd.get_dummies(df_1["Marital_Status_clean"],
                                  prefix="Marital",
                                  drop_first=True)

df_1 = pd.concat([df_1, marital_dummies], axis=1)

COLS_CAT_ENC = ["Education_enc"] + list(marital_dummies.columns)
print(f"\nColumnas categóricas codificadas: {COLS_CAT_ENC}")

# --- Escalar las categóricas codificadas ---
scaler_cat   = StandardScaler()
X_cat_scaled = scaler_cat.fit_transform(df_1[COLS_CAT_ENC].values)

# --- Concatenar con X_scaled ---
X_full_scaled    = np.hstack([X_scaled, X_cat_scaled])
ALL_FEATURE_NAMES = COLS_NUMERICAS + COLS_CAT_ENC

print(f"\nDimensionalidad final antes de PCA: {X_full_scaled.shape[1]} variables")
print(f"Shape de X_full_scaled: {X_full_scaled.shape}")

In [ ]:
# =============================================================================
# CELDA 9 · PCA para reducción de dimensionalidad → X_pca
# Se ajusta PCA explorando hasta 15 componentes para tener visibilidad completa.
# Si el umbral del 85% no es alcanzable en el rango 5-10, se documenta
# y se selecciona el número de componentes que maximice varianza en ese rango.
# =============================================================================

N_EXPLORE = min(15, X_full_scaled.shape[1])   # explorar hasta 15 componentes

pca_full = PCA(n_components=N_EXPLORE, random_state=RANDOM_STATE)
pca_full.fit(X_full_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

# --- Diagnóstico del umbral ---
var_en_5  = cumvar[4]    # varianza acumulada con 5 componentes
var_en_10 = cumvar[9]    # varianza acumulada con 10 componentes

print(f"Varianza acumulada con  5 componentes: {var_en_5:.1%}")
print(f"Varianza acumulada con 10 componentes: {var_en_10:.1%}")

umbral_alcanzable = cumvar[PCA_N_MAX - 1] >= PCA_VAR_TARGET

if umbral_alcanzable:
    n_components_opt = int(np.argmax(cumvar >= PCA_VAR_TARGET)) + 1
    n_components_opt = max(PCA_N_MIN, min(n_components_opt, PCA_N_MAX))
    nota_pca = f"{n_components_opt} componentes alcanzan el umbral del {PCA_VAR_TARGET:.0%}."
else:
    # El umbral del 85% no es alcanzable en el rango 5-10 con este dataset.
    # Se selecciona el máximo del rango (10) para maximizar varianza capturada.
    # Esta limitación se documenta en la celda Markdown siguiente.
    n_components_opt = PCA_N_MAX
    nota_pca = (f"El umbral del {PCA_VAR_TARGET:.0%} no se alcanza en el rango "
                f"{PCA_N_MIN}-{PCA_N_MAX} (máximo: {var_en_10:.1%} con {PCA_N_MAX} componentes). "
                f"Se seleccionan {PCA_N_MAX} componentes para maximizar varianza capturada.")

print(f"\n{nota_pca}")

# --- Ajustar PCA final ---
pca = PCA(n_components=n_components_opt, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_full_scaled)
var_capturada = cumvar[n_components_opt - 1]

print(f"\nComponentes seleccionadas : {n_components_opt}")
print(f"Varianza capturada        : {var_capturada:.1%}")
print(f"Shape de X_pca            : {X_pca.shape}")

# --- Visualización ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Varianza por componente
axes[0].bar(range(1, N_EXPLORE + 1),
            pca_full.explained_variance_ratio_ * 100,
            color=CLUSTER_PALETTE[0], edgecolor="white")
axes[0].set_title("Varianza explicada por componente")
axes[0].set_xlabel("Componente principal")
axes[0].set_ylabel("Varianza explicada (%)")
axes[0].set_xticks(range(1, N_EXPLORE + 1))

# Varianza acumulada
axes[1].plot(range(1, N_EXPLORE + 1), cumvar * 100,
             marker="o", color=CLUSTER_PALETTE[1], linewidth=2)
axes[1].axhline(PCA_VAR_TARGET * 100, color="red", linestyle="--",
                linewidth=1.2, label=f"Umbral {PCA_VAR_TARGET:.0%}")
axes[1].axvline(n_components_opt, color="gray", linestyle=":",
                linewidth=1.2, label=f"Seleccionadas: {n_components_opt}")
axes[1].set_title("Varianza acumulada explicada")
axes[1].set_xlabel("Número de componentes")
axes[1].set_ylabel("Varianza acumulada (%)")
axes[1].set_xticks(range(1, N_EXPLORE + 1))
axes[1].legend()

plt.suptitle("Análisis de Componentes Principales (PCA)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELDA 10 · Resumen del preprocesamiento
# =============================================================================

resumen = pd.DataFrame({
    "Etapa"       : ["Dataset original", "Tras imputación (df_0)",
                      "Variables numéricas", "Tras codif. categóricas",
                      "X_full_scaled", "X_pca"],
    "Filas"       : [df.shape[0], df_0.shape[0],
                      df_0.shape[0], df_1.shape[0],
                      X_full_scaled.shape[0], X_pca.shape[0]],
    "Columnas"    : [df.shape[1], df_0.shape[1],
                      len(COLS_NUMERICAS), len(ALL_FEATURE_NAMES),
                      X_full_scaled.shape[1], X_pca.shape[1]],
    "Descripción" : [
        "Sin modificar",
        f"Income imputado con mediana ({income_median:,.0f} €); 3 edades >90 corregidas",
        "16 variables numéricas (incluye Age derivada de Year_Birth)",
        f"Marital_Status limpiada (Alone/YOLO→Single); OHE + Education ordinal",
        "Todo escalado con StandardScaler",
        # La línea siguiente refleja la realidad: si el umbral no se alcanzó, lo dice
        f"{n_components_opt} componentes PCA — varianza capturada: {var_capturada:.1%}"
        + ("" if umbral_alcanzable else " (umbral 85% no alcanzable en rango 5-10)")
    ]
})

display(resumen.style.set_caption("Pipeline de preprocesamiento"))

In [ ]:
# =============================================================================
# CELDA 11 · Búsqueda del k óptimo para K-Means (k = 2..10)
# Se evalúan tres métricas complementarias sobre dos espacios de features:
#   · X_pca        (10 componentes, 79.8% varianza)
#   · X_full_scaled (21 variables escaladas, sin reducción)
# Comparar ambos permite decidir si PCA mejora la calidad del clustering.
# =============================================================================

resultados = {espacio: [] for espacio in ["pca", "full"]}
espacios   = {"pca": X_pca, "full": X_full_scaled}

for nombre, X in espacios.items():
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(X)

        inercia  = km.inertia_
        silueta  = silhouette_score(X, labels, random_state=RANDOM_STATE)
        db_index = davies_bouldin_score(X, labels)

        resultados[nombre].append({
            "k"       : k,
            "inercia" : inercia,
            "silueta" : silueta,
            "db_index": db_index,
        })

df_metricas_pca  = pd.DataFrame(resultados["pca"])
df_metricas_full = pd.DataFrame(resultados["full"])

print("=== Métricas K-Means sobre X_pca ===")
display(df_metricas_pca.style
        .format({"inercia": "{:,.1f}", "silueta": "{:.4f}", "db_index": "{:.4f}"})
        .highlight_max(subset=["silueta"], color="#c6efce")
        .highlight_min(subset=["db_index", "inercia"], color="#c6efce"))

print("\n=== Métricas K-Means sobre X_full_scaled ===")
display(df_metricas_full.style
        .format({"inercia": "{:,.1f}", "silueta": "{:.4f}", "db_index": "{:.4f}"})
        .highlight_max(subset=["silueta"], color="#c6efce")
        .highlight_min(subset=["db_index", "inercia"], color="#c6efce"))

In [ ]:
# =============================================================================
# CELDA 11b · Re-evaluación usando solo variables numéricas de comportamiento
# La inclusión de variables categóricas codificadas (Marital_Status, Education)
# distorsiona el clustering: K-Means agrupa por demografía en lugar de por
# comportamiento de compra, que es el objetivo de negocio.
# Se re-evalúa usando únicamente X_scaled (16 variables numéricas).
# =============================================================================

resultados_num = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)

    resultados_num.append({
        "k"       : k,
        "inercia" : km.inertia_,
        "silueta" : silhouette_score(X_scaled, labels,
                                     random_state=RANDOM_STATE),
        "db_index": davies_bouldin_score(X_scaled, labels),
    })

df_metricas_num = pd.DataFrame(resultados_num)

print("=== Métricas K-Means sobre X_scaled (solo numéricas) ===")
display(df_metricas_num.style
        .format({"inercia": "{:,.1f}", "silueta": "{:.4f}", "db_index": "{:.4f}"})
        .highlight_max(subset=["silueta"], color="#c6efce")
        .highlight_min(subset=["db_index", "inercia"], color="#c6efce"))

In [ ]:
# =============================================================================
# CELDA 12 · Visualización de métricas para selección de k
# Se muestran los tres espacios evaluados para justificar la elección final.
# =============================================================================

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle("Selección de k óptimo — K-Means (comparativa de espacios)",
             fontsize=15, fontweight="bold")

titulos  = ["X_pca (10 comp.)", "X_full_scaled (21 vars.)", "X_scaled (16 numéricas)"]
datasets = [df_metricas_pca, df_metricas_full, df_metricas_num]
colores  = [CLUSTER_PALETTE[0], CLUSTER_PALETTE[2], CLUSTER_PALETTE[3]]

for fila, (df_m, titulo, color) in enumerate(zip(datasets, titulos, colores)):

    # --- Método del codo ---
    axes[fila, 0].plot(df_m["k"], df_m["inercia"],
                       marker="o", color=color, linewidth=2)
    axes[fila, 0].set_title(f"Codo — {titulo}")
    axes[fila, 0].set_xlabel("k")
    axes[fila, 0].set_ylabel("Inercia (WCSS)")
    axes[fila, 0].set_xticks(list(K_RANGE))

    # --- Silueta ---
    k_mejor_sil = df_m.loc[df_m["silueta"].idxmax(), "k"]
    axes[fila, 1].plot(df_m["k"], df_m["silueta"],
                       marker="o", color=color, linewidth=2)
    axes[fila, 1].axvline(k_mejor_sil, color="red", linestyle="--",
                          linewidth=1.2, label=f"Mejor k={k_mejor_sil}")
    axes[fila, 1].set_title(f"Silueta — {titulo}")
    axes[fila, 1].set_xlabel("k")
    axes[fila, 1].set_ylabel("Silueta")
    axes[fila, 1].set_xticks(list(K_RANGE))
    axes[fila, 1].legend()

    # --- Davies-Bouldin ---
    k_mejor_db = df_m.loc[df_m["db_index"].idxmin(), "k"]
    axes[fila, 2].plot(df_m["k"], df_m["db_index"],
                       marker="o", color=color, linewidth=2)
    axes[fila, 2].axvline(k_mejor_db, color="red", linestyle="--",
                          linewidth=1.2, label=f"Mejor k={k_mejor_db}")
    axes[fila, 2].set_title(f"Davies-Bouldin — {titulo}")
    axes[fila, 2].set_xlabel("k")
    axes[fila, 2].set_ylabel("DB Index")
    axes[fila, 2].set_xticks(list(K_RANGE))
    axes[fila, 2].legend()

plt.tight_layout()
plt.show()

## Selección del k óptimo — K-Means

### Comparativa de espacios de features

Se evaluaron tres espacios de representación para determinar cuál produce 
clusters de mayor calidad:

| Espacio | Silueta k=2 | Silueta k=4 | DB k=2 | Observación |
|---------|------------|------------|--------|-------------|
| X_pca (10 comp.) | 0.272 | 0.170 | 1.510 | Clusters dominados por demografía |
| X_full_scaled (21 vars.) | 0.234 | 0.163 | 1.765 | Ídem, peor que PCA |
| X_scaled (16 numéricas) | 0.302 | 0.163 | 1.449 | Mejor separación, clusters coherentes |

**Espacio seleccionado: X_scaled (16 variables numéricas)**

La inclusión de variables categóricas codificadas (Marital_Status, Education) 
introdujo un artefacto: K-Means formó un cluster compuesto en un 92% por clientes 
con estado civil "Together", agrupando por demografía en lugar de por comportamiento 
de compra. Al eliminar las categóricas, los clusters reflejan patrones reales de 
gasto y frecuencia de compra, que es el objetivo de negocio.

Las variables categóricas se usan únicamente como descriptores post-hoc para 
caracterizar cada cluster, no como input del algoritmo.

### Resumen de métricas sobre X_scaled

| Criterio | k sugerido | Interpretación |
|----------|-----------|----------------|
| Método del codo | Sin codo claro | La inercia decrece continuamente sin inflexión nítida |
| Silueta máxima | k=2 (0.302) | Clusters más compactos y separados estadísticamente |
| Davies-Bouldin mínimo | k=2 (1.449) | Menor solapamiento entre clusters |

### k elegido: 4

Las métricas apuntan a k=2 como óptimo estadístico. Sin embargo, **k=2 es 
operativamente insuficiente** para el objetivo de negocio: dos segmentos no 
permiten diseñar campañas diferenciadas ni asignar recursos promocionales 
de forma granular.

Se elige **k=4** por las siguientes razones:

- **Técnica:** la silueta de k=4 (0.163) no supone una degradación 
  catastrófica respecto a k=2 (0.302); la separación entre clusters 
  sigue siendo positiva en todos los grupos.
- **Negocio:** 4 perfiles permiten al equipo de marketing diseñar 
  estrategias distintas por segmento (canal, producto, frecuencia, 
  descuento) sin fragmentación excesiva que dificulte la operativa.
- **Precedente:** la segmentación en 4 grupos es ampliamente utilizada 
  en retail para capturar los ejes principales de diferenciación: 
  nivel de gasto, frecuencia y canal preferido.
- **Enunciado:** el propio enunciado plantea explícitamente elegir un k 
  alternativo con justificación de negocio suficiente aunque las métricas 
  sean ligeramente peores.

In [ ]:
# =============================================================================
# CELDA 14 · Modelo final K-Means con k=4 sobre X_scaled
# Se usa el espacio de 16 variables numéricas que produce clusters coherentes
# con el comportamiento de compra, evitando el artefacto demográfico detectado
# al incluir variables categóricas codificadas.
# =============================================================================

K_OPTIMO = 4

km_final = KMeans(n_clusters=K_OPTIMO, random_state=RANDOM_STATE, n_init=10)
km_final.fit(X_scaled)

df_1["Cluster_KMeans"] = km_final.labels_

# --- Tamaño de clusters ---
conteo     = df_1["Cluster_KMeans"].value_counts().sort_index()
conteo_pct = (conteo / len(df_1) * 100).round(1)

resumen_clusters = pd.DataFrame({
    "Cluster"    : conteo.index,
    "N clientes" : conteo.values,
    "% del total": conteo_pct.values
})

print(f"K-Means final — k={K_OPTIMO} sobre X_scaled")
print(f"Silueta:        {silhouette_score(X_scaled, km_final.labels_, random_state=RANDOM_STATE):.4f}")
print(f"Davies-Bouldin: {davies_bouldin_score(X_scaled, km_final.labels_):.4f}")
print()
display(resumen_clusters.style.format({"% del total": "{:.1f}%"}))

# --- Visualización sobre PC1 y PC2 (solo para proyección visual) ---
# X_pca se usa aquí únicamente como espacio de visualización 2D;
# el modelo fue entrenado sobre X_scaled.
fig, ax = plt.subplots(figsize=(9, 6))

for cluster_id in range(K_OPTIMO):
    mask = km_final.labels_ == cluster_id
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[CLUSTER_PALETTE[cluster_id]],
               label=f"Cluster {cluster_id}",
               alpha=0.6, s=30, edgecolors="none")

ax.set_title("K-Means (k=4, X_scaled) — Proyección sobre PC1 y PC2")
ax.set_xlabel("Componente Principal 1")
ax.set_ylabel("Componente Principal 2")
ax.legend(title="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELDA 15 · Centroides en escala original mediante inverse_transform
# Los centroides están en el espacio escalado (X_scaled).
# inverse_transform recupera las unidades originales (euros, días, unidades).
# Las variables categóricas se analizan por distribución de frecuencias,
# ya que no formaron parte del espacio de clustering.
# =============================================================================

centroides_scaled = km_final.cluster_centers_              # shape (4, 16)
centroides_orig   = scaler.inverse_transform(centroides_scaled)  # shape (4, 16)

df_centroides = pd.DataFrame(
    centroides_orig,
    columns=COLS_NUMERICAS,
    index=[f"Cluster {i}" for i in range(K_OPTIMO)]
).T

print("Centroides en escala original (variables numéricas)")
display(df_centroides.style.format("{:.1f}"))

# --- Distribución de categóricas por cluster (descriptores post-hoc) ---
print("\nDistribución de Education por cluster (%):")
edu_dist = (df_1.groupby("Cluster_KMeans")["Education"]
              .value_counts(normalize=True)
              .mul(100).round(1)
              .unstack(fill_value=0))
display(edu_dist.style.format("{:.1f}%"))

print("\nDistribución de Marital_Status por cluster (%):")
mar_dist = (df_1.groupby("Cluster_KMeans")["Marital_Status_clean"]
              .value_counts(normalize=True)
              .mul(100).round(1)
              .unstack(fill_value=0))
display(mar_dist.style.format("{:.1f}%"))

In [ ]:
# =============================================================================
# CELDA 16 · Interpretación de clusters y estrategias de marketing
# Los centroides se analizan en unidades originales para extraer perfiles
# de negocio accionables. El heatmap normaliza cada variable entre 0 y 1
# para comparar el comportamiento relativo entre clusters visualmente.
# =============================================================================

# --- Heatmap de centroides normalizados con valores originales anotados ---
df_heatmap = df_centroides.copy()

# Normalización min-max por fila (variable): permite comparar clusters
# independientemente de la escala de cada variable.
row_min = df_heatmap.min(axis=1).values.reshape(-1, 1)
row_max = df_heatmap.max(axis=1).values.reshape(-1, 1)
df_norm = (df_heatmap - row_min) / (row_max - row_min + 1e-9)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_norm, annot=df_centroides.round(1), fmt="g",
            cmap="YlOrRd", ax=ax, linewidths=0.5,
            cbar_kws={"label": "Valor relativo (0 = mínimo, 1 = máximo)"})
ax.set_title("Perfil de centroides por cluster\n(color relativo · valores anotados en unidades originales)",
             fontsize=13)
ax.set_xlabel("Cluster")
ax.set_ylabel("Variable")
plt.tight_layout()
plt.show()

## Interpretación de clusters y estrategias de marketing

---

### Cluster 0 — "Familias jóvenes de bajo presupuesto"

**Perfil:**
- El cluster más joven (edad media: 36 años) y de menor renta (30.000 €/año).
- Gasto muy bajo en todas las categorías de producto (vino: 36 €, carne: 28 €).
- Núcleo familiar con hijos pequeños (Kidhome ≈ 0.9, el más alto).
- Sin adolescentes en casa; compran principalmente en tienda física y web básica.
- Nivel educativo más bajo del conjunto (casi 8% con educación básica).

**Interpretación:** clientes en fase de formación de familia con restricciones 
presupuestarias claras. El gasto se destina a necesidades básicas, no a productos 
premium ni compras por catálogo.

**Estrategia de marketing:**
- Promociones en productos de gran consumo (lácteos, carne básica, frutas).
- Programas de fidelización con descuentos acumulativos por volumen.
- Comunicación por web y app móvil (canal preferido por perfil joven).
- Ofertas de "pack familiar" con ahorro visible en el ticket.

---

### Cluster 1 — "Compradores premium de alto poder adquisitivo"

**Perfil:**
- Mayor renta del dataset (76.500 €/año) y gasto más alto en todas las categorías.
- Gasto en vino (583 €) y carne (454 €) muy por encima de la media.
- Sin hijos pequeños en casa (Kidhome ≈ 0.0); perfil maduro (edad media: 46 años).
- Usan todos los canales: tienda física (8.4 compras), catálogo (6.0) y web (5.2).
- Nivel educativo alto: 20% con doctorado.
- Apenas usan descuentos (NumDealsPurchases: 1.3, el más bajo).

**Interpretación:** cliente de alto valor económico, fidelizado y poco sensible 
al precio. Valora la calidad y la variedad; compra por todos los canales disponibles.

**Estrategia de marketing:**
- Programa de cliente VIP con acceso anticipado a productos premium y nuevas líneas.
- Catálogo físico de alta calidad con selección de vinos y productos gourmet.
- Invitaciones a catas, eventos gastronómicos o experiencias exclusivas en tienda.
- Comunicación personalizada y de alta frecuencia; toleran bien el contacto directo.

---

### Cluster 2 — "Familias con adolescentes de renta media-baja"

**Perfil:**
- Renta media-baja (42.400 €/año), gasto muy reducido en todas las categorías.
- El cluster de mayor edad media junto con Cluster 3 (50 años).
- Perfil familiar con adolescentes en casa (Teenhome ≈ 1.0, el más alto del dataset).
- Mayor uso de descuentos (NumDealsPurchases: 2.5) y visitas web (5.9/mes).
- Compras principalmente en tienda física; catálogo prácticamente nulo (0.7).
- Nivel educativo relativamente alto (24% con Máster o PhD).

**Interpretación:** familias en fase de hijos adolescentes con presupuesto ajustado. 
Buscan activamente ofertas y descuentos. Son fieles a la tienda física pero 
consultan la web para comparar precios antes de comprar.

**Estrategia de marketing:**
- Campañas de descuento activo y comunicación de ofertas semanales por email.
- Promociones en productos orientados a adolescentes (snacks, bebidas, cereales).
- Newsletter digital con comparativa de precios y productos en oferta.
- Tarjeta de puntos con ventajas tangibles a corto plazo (no diferidas).

---

### Cluster 3 — "Compradores activos de renta media-alta"

**Perfil:**
- Renta media-alta (59.600 €/año); gasto significativo en vino (528 €) y carne (147 €).
- Mayor uso de descuentos del dataset (NumDealsPurchases: 3.7) pese a buena renta.
- Mayor número de compras web (6.6, el más alto) y catálogo (3.3).
- Sin hijos pequeños; adolescentes en casa (Teenhome ≈ 0.9).
- Nivel educativo alto (29% con doctorado, el más alto del dataset).
- Edad media: 49 años; estado civil mixto sin patrón dominante.

**Interpretación:** cliente con capacidad económica que aun así maximiza el valor 
de sus compras buscando descuentos. Es el perfil más activo digitalmente y más 
receptivo a ofertas multicanal. Combina gasto alto con comportamiento de "smart shopper".

**Estrategia de marketing:**
- Ofertas personalizadas basadas en historial de compra web y catálogo.
- Descuentos por volumen en categorías de alto gasto (vino, carne).
- Programa de referidos con incentivo económico directo.
- Comunicación omnicanal: web, catálogo y tienda con coherencia de precio.

---

### Resumen comparativo

| | Cluster 0 | Cluster 1 | Cluster 2 | Cluster 3 |
|---|---|---|---|---|
| **Nombre** | Familias jóvenes bajo presupuesto | Compradores premium | Familias con adolescentes | Smart shoppers activos |
| **Renta media** | 30.000 € | 76.500 € | 42.400 € | 59.600 € |
| **Gasto total aprox.** | ~105 € | ~1.354 € | ~125 € | ~816 € |
| **Canal principal** | Tienda + web básica | Todos los canales | Tienda física | Web + catálogo |
| **Sensibilidad precio** | Alta | Baja | Alta | Media-alta |
| **Tamaño** | 34.1% | 25.2% | 14.8% | 26.0% |

In [ ]:
# =============================================================================
# CELDA 17 · Clustering jerárquico aglomerativo — linkage Ward
# Ward minimiza la varianza intra-cluster en cada fusión, lo que tiende
# a producir clusters de tamaño equilibrado. Es el criterio más comparable
# con K-Means porque ambos minimizan varianza interna.
# El dendrograma se trunca a las últimas 50 fusiones para que sea legible;
# con 2240 observaciones el árbol completo es visualmente ilegible.
# =============================================================================

# --- Dendrograma (sobre X_scaled, mismo espacio que K-Means) ---
fig, ax = plt.subplots(figsize=(14, 5))

linkage_matrix = linkage(X_scaled, method="ward")

dendrogram(
    linkage_matrix,
    truncate_mode="lastp",   # mostrar solo las últimas p fusiones
    p=50,                    # las 50 fusiones más altas del árbol
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
    color_threshold=linkage_matrix[-(K_OPTIMO - 1), 2],  # línea de corte en k=4
)

# Línea de corte que indica dónde se obtienen k=4 clusters
corte_altura = linkage_matrix[-(K_OPTIMO - 1), 2]
ax.axhline(corte_altura, color="red", linestyle="--", linewidth=1.5,
           label=f"Corte en k={K_OPTIMO} (altura={corte_altura:.1f})")

ax.set_title(f"Dendrograma — Linkage Ward (truncado a últimas 50 fusiones)")
ax.set_xlabel("Observaciones (o tamaño del grupo entre paréntesis)")
ax.set_ylabel("Distancia de fusión")
ax.legend()
plt.tight_layout()
plt.show()

# --- Modelo final jerárquico con k=4 y Ward ---
hier_ward = AgglomerativeClustering(n_clusters=K_OPTIMO, linkage="ward")
labels_ward = hier_ward.fit_predict(X_scaled)

df_1["Cluster_Hier_Ward"] = labels_ward

sil_ward = silhouette_score(X_scaled, labels_ward, random_state=RANDOM_STATE)
db_ward  = davies_bouldin_score(X_scaled, labels_ward)

print(f"Clustering jerárquico — Ward, k={K_OPTIMO}")
print(f"Silueta:        {sil_ward:.4f}")
print(f"Davies-Bouldin: {db_ward:.4f}")
print()

# Tamaño de clusters
conteo_ward = pd.Series(labels_ward).value_counts().sort_index()
resumen_ward = pd.DataFrame({
    "Cluster"    : conteo_ward.index,
    "N clientes" : conteo_ward.values,
    "% del total": (conteo_ward / len(df_1) * 100).round(1).values
})
display(resumen_ward.style.format({"% del total": "{:.1f}%"}))

## Interpretación del dendrograma — Linkage Ward

El dendrograma representa el proceso de fusión jerárquica de clusters.
Cada unión en el árbol indica en qué momento (a qué distancia) dos grupos
se fusionaron en uno mayor.

**Qué información proporciona:**

- **Eje Y (altura de fusión):** distancias a las que se producen las fusiones.
  Saltos grandes indican que los grupos que se fusionan son muy distintos entre sí;
  saltos pequeños indican grupos muy similares.
- **Estructura del árbol:** permite identificar subestructuras naturales en los datos.
  Si hay un salto grande antes del corte en k=4, los 4 clusters son relativamente
  bien separados.
- **Línea de corte roja:** fijada a la altura que produce exactamente k=4 clusters,
  consistente con la elección realizada en K-Means.

**Observaciones sobre este dendrograma:**

- Las fusiones finales (parte superior del árbol) se producen a alturas elevadas,
  lo que indica que los 4 grandes grupos son sustancialmente distintos entre sí.
- La ausencia de un único salto dominante confirma lo observado en K-Means:
  los datos no presentan una estructura de clusters perfectamente separada,
  sino una separación gradual y continua.
- Los cuatro clusters resultantes tienen tamaños equilibrados (486–608 clientes),
  lo que es coherente con el criterio Ward, que penaliza las fusiones que
  incrementan la varianza interna.
- El truncado a las 50 últimas fusiones es necesario porque con 2240 observaciones
  el árbol completo es visualmente ilegible; cada hoja representa un subgrupo
  cuyo tamaño aparece entre paréntesis.

In [ ]:
# =============================================================================
# CELDA 19 · Comparativa de criterios de enlace: Ward vs complete vs average
# Cada criterio define de forma distinta la distancia entre clusters:
#   · Ward:     minimiza la varianza intra-cluster al fusionar (más compacto)
#   · Complete: distancia máxima entre puntos de dos clusters (más conservador)
#   · Average:  distancia media entre todos los pares de puntos (intermedio)
# Se comparan con silueta y Davies-Bouldin sobre el mismo espacio X_scaled.
# =============================================================================

linkages_a_probar = ["ward", "complete", "average"]
resultados_hier   = []

for lnk in linkages_a_probar:
    modelo  = AgglomerativeClustering(n_clusters=K_OPTIMO, linkage=lnk)
    labels  = modelo.fit_predict(X_scaled)

    sil = silhouette_score(X_scaled, labels, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X_scaled, labels)

    conteo = pd.Series(labels).value_counts().sort_index()

    resultados_hier.append({
        "Linkage"        : lnk,
        "Silueta"        : sil,
        "Davies-Bouldin" : db,
        "Tamaño clusters": conteo.values.tolist(),
    })

df_hier = pd.DataFrame(resultados_hier)

print(f"Comparativa de linkages — Clustering jerárquico (k={K_OPTIMO})")
display(df_hier.style
        .format({"Silueta": "{:.4f}", "Davies-Bouldin": "{:.4f}"})
        .highlight_max(subset=["Silueta"], color="#c6efce")
        .highlight_min(subset=["Davies-Bouldin"], color="#c6efce"))

# --- Visualización: silueta por linkage ---
fig, ax = plt.subplots(figsize=(7, 4))

colores_lnk = [CLUSTER_PALETTE[i] for i in range(len(linkages_a_probar))]
bars = ax.bar(df_hier["Linkage"], df_hier["Silueta"],
              color=colores_lnk, edgecolor="white", width=0.5)

ax.bar_label(bars, fmt="{:.4f}", padding=3, fontsize=10)
ax.set_title(f"Coeficiente de silueta por criterio de enlace (k={K_OPTIMO})")
ax.set_xlabel("Criterio de enlace (linkage)")
ax.set_ylabel("Silueta (mayor = mejor)")
ax.set_ylim(0, df_hier["Silueta"].max() * 1.2)
plt.tight_layout()
plt.show()

## Comparativa de criterios de enlace

| Linkage  | Silueta | Davies-Bouldin | Tamaños de clusters        |
|----------|---------|----------------|---------------------------|
| Ward     | 0.1375  | 1.9038         | [588, 558, 608, 486]      |
| Complete | 0.5428  | 0.6754         | [2231, 4, 4, 1]           |
| Average  | 0.5428  | 0.6754         | [2231, 4, 4, 1]           |

### Interpretación crítica

A primera vista, complete y average parecen superiores (silueta 0.54 vs 0.14).
Sin embargo, sus tamaños revelan un **cluster degenerado**: 2231 de los 2240
clientes quedan en un único macro-grupo, mientras que los tres clusters restantes
contienen 4, 4 y 1 observaciones respectivamente.

Este fenómeno se produce porque complete y average son muy sensibles a outliers.
Los clientes de renta o gasto extremadamente alto (detectados en la FASE 1)
se separan como clusters aislados, y el coeficiente de silueta sube
artificialmente porque esos puntos están muy lejos del macro-grupo.
La silueta alta es un **artefacto estadístico**, no una segmentación útil.

**Criterio ganador operativo: Ward**

Ward es el único linkage que produce una segmentación accionable para negocio.
Sus clusters tienen tamaños equilibrados (21.7%–27.1%) y la silueta, aunque
más baja (0.1375), refleja una separación real entre grupos heterogéneos.

### Diferencias en la forma de los clusters

- **Ward** penaliza las fusiones que aumentan la varianza interna, favoreciendo
  clusters compactos y de tamaño similar. Es robusto frente a outliers moderados.
- **Complete** define la distancia entre clusters como la distancia máxima entre
  sus puntos. Es muy conservador: evita fusionar grupos con puntos lejanos aunque
  el resto sea similar, lo que provoca que outliers extremos formen clusters propios.
- **Average** usa la distancia media entre todos los pares de puntos. Comportamiento
  intermedio en teoría, pero con outliers extremos produce el mismo efecto que
  complete, como se observa en este dataset.

### Comparación con K-Means

El clustering jerárquico con Ward produce resultados similares a K-Means
en cuanto a estructura (4 grupos equilibrados), pero con silueta ligeramente
inferior (0.1375 vs 0.1634). Esto es esperable: K-Means optimiza directamente
la compacidad de los clusters, mientras que Ward optimiza la varianza en cada
paso de fusión sin visión global. La distribución equilibrada de tamaños en
ambos algoritmos confirma que la estructura de 4 grupos es estable y no
un artefacto de la inicialización.

In [ ]:
# =============================================================================
# CELDA 21 · Curvas k-distance para orientar la selección de eps
# La curva k-distance ordena las distancias al k-ésimo vecino más cercano
# de menor a mayor. El "codo" de la curva indica el valor de eps a partir
# del cual los puntos empiezan a considerarse ruido.
# Se genera una curva por cada valor de min_samples a explorar.
# =============================================================================

from sklearn.neighbors import NearestNeighbors

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Curvas k-distance por min_samples\n"
             "(el codo orienta el valor de eps)", fontsize=13, fontweight="bold")

axes = axes.flatten()

for idx, ms in enumerate(DBSCAN_MIN_SAMPLES):
    # Ajustar vecinos más cercanos con k = min_samples
    nbrs = NearestNeighbors(n_neighbors=ms).fit(X_scaled)
    distancias, _ = nbrs.kneighbors(X_scaled)

    # Distancia al vecino más lejano (el k-ésimo) para cada punto
    dist_k = np.sort(distancias[:, -1])

    axes[idx].plot(dist_k, color=CLUSTER_PALETTE[idx], linewidth=1.5)
    axes[idx].axhline(y=0.5, color="gray", linestyle=":", linewidth=1,
                      label="eps=0.5 (referencia)")
    axes[idx].axhline(y=1.0, color="red", linestyle="--", linewidth=1,
                      label="eps=1.0 (referencia)")
    axes[idx].set_title(f"min_samples = {ms}")
    axes[idx].set_xlabel("Puntos ordenados por distancia")
    axes[idx].set_ylabel(f"Distancia al vecino {ms}")
    axes[idx].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELDA 22 · Grid de parámetros DBSCAN
# Se exploran todas las combinaciones de min_samples y eps definidas en
# la celda de configuración. Para cada combinación se registran:
#   · n_clusters: grupos encontrados (excluyendo ruido, etiqueta -1)
#   · n_ruido: puntos etiquetados como ruido
#   · pct_ruido: porcentaje de ruido sobre el total
#   · silueta: calculada SOLO sobre puntos no-ruido (requisito del enunciado)
#              si todos son ruido o hay un solo cluster, se registra NaN
# =============================================================================

resultados_dbscan = []

for ms in DBSCAN_MIN_SAMPLES:
    for eps in DBSCAN_EPS_RANGE:
        db     = DBSCAN(eps=eps, min_samples=ms)
        labels = db.fit_predict(X_scaled)

        n_ruido    = (labels == -1).sum()
        pct_ruido  = n_ruido / len(labels) * 100
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        # Silueta solo sobre puntos no-ruido y solo si hay ≥2 clusters
        mask_no_ruido = labels != -1
        if n_clusters >= 2 and mask_no_ruido.sum() > 0:
            sil = silhouette_score(X_scaled[mask_no_ruido],
                                   labels[mask_no_ruido],
                                   random_state=RANDOM_STATE)
        else:
            sil = np.nan

        resultados_dbscan.append({
            "min_samples" : ms,
            "eps"         : eps,
            "n_clusters"  : n_clusters,
            "n_ruido"     : n_ruido,
            "pct_ruido"   : pct_ruido,
            "silueta"     : sil,
        })

df_dbscan = pd.DataFrame(resultados_dbscan)

# --- Tabla resumen ---
print(f"Grid DBSCAN: {len(df_dbscan)} combinaciones exploradas")
print(f"Combinaciones con <50% ruido: "
      f"{(df_dbscan['pct_ruido'] < 50).sum()}")
print(f"Combinaciones con ≥2 clusters y silueta válida: "
      f"{df_dbscan['silueta'].notna().sum()}")
print()
display(df_dbscan.style
        .format({"eps": "{:.2f}", "pct_ruido": "{:.1f}%",
                 "silueta": "{:.4f}"})
        .highlight_min(subset=["pct_ruido"], color="#c6efce")
        .highlight_max(subset=["silueta"], color="#ffd700",
                       props="color:black"))

In [ ]:
# =============================================================================
# CELDA 23 · Visualización del mejor resultado DBSCAN
# Se selecciona la combinación representativa según este criterio:
#   1. Si existe alguna con <50% ruido y silueta válida → la de mayor silueta
#   2. Si no → la de menor porcentaje de ruido (más conservadora)
# La visualización proyecta sobre PC1-PC2 (solo para representación 2D).
# =============================================================================

# --- Selección de la combinación representativa ---
df_validas = df_dbscan[df_dbscan["pct_ruido"] < 50].dropna(subset=["silueta"])

if len(df_validas) > 0:
    mejor = df_validas.loc[df_validas["silueta"].idxmax()]
    criterio = "mejor silueta con <50% ruido"
else:
    # No hay combinación con <50% ruido: elegir la de menor ruido
    mejor = df_dbscan.loc[df_dbscan["pct_ruido"].idxmin()]
    criterio = "menor porcentaje de ruido (ninguna combinación supera el umbral del 50%)"

eps_rep = mejor["eps"]
ms_rep  = int(mejor["min_samples"])

print(f"Combinación seleccionada ({criterio}):")
print(f"  eps={eps_rep:.2f}, min_samples={ms_rep}")
print(f"  Clusters: {int(mejor['n_clusters'])} | "
      f"Ruido: {int(mejor['n_ruido'])} ({mejor['pct_ruido']:.1f}%) | "
      f"Silueta: {mejor['silueta'] if pd.notna(mejor['silueta']) else 'N/A'}")

# --- Reentrenar con la combinación seleccionada ---
db_rep     = DBSCAN(eps=eps_rep, min_samples=ms_rep)
labels_rep = db_rep.fit_predict(X_scaled)

# --- Scatter sobre PC1 y PC2 ---
fig, ax = plt.subplots(figsize=(9, 6))

cluster_ids = sorted(set(labels_rep))
for cid in cluster_ids:
    mask  = labels_rep == cid
    color = NOISE_COLOR if cid == -1 else CLUSTER_PALETTE[cid % len(CLUSTER_PALETTE)]
    label = "Ruido (-1)" if cid == -1 else f"Cluster {cid}"
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[color], label=label,
               alpha=0.5, s=25, edgecolors="none")

ax.set_title(f"DBSCAN (eps={eps_rep:.2f}, min_samples={ms_rep})\n"
             f"Proyección sobre PC1 y PC2")
ax.set_xlabel("Componente Principal 1")
ax.set_ylabel("Componente Principal 2")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## DBSCAN — Análisis crítico de adecuación del algoritmo

### Resultados del grid de parámetros

Se exploraron 40 combinaciones (min_samples ∈ {3, 5, 8, 10} ×
eps ∈ [0.30, 2.50], 10 valores equidistantes).

| Resultado | Valor |
|-----------|-------|
| Combinaciones con <50% ruido | 18 de 40 |
| Combinaciones con silueta válida (≥2 clusters) | 31 de 40 |
| Mejor combinación por silueta con <50% ruido | eps=2.01, min_samples=10 |
| Ruido en mejor combinación | 903 puntos (40.3%) |
| Silueta en mejor combinación | 0.2297 |

### Interpretación de las curvas k-distance

Las curvas k-distance muestran un codo claro para todos los valores de
min_samples explorados. Esto indica que **sí existe estructura de densidad**
en los datos: hay una transición nítida entre puntos en zonas densas y puntos
aislados. Sin embargo, como se verá a continuación, esa estructura no
corresponde a segmentos de negocio útiles.

### Análisis de los resultados del grid

La combinación con mejor silueta válida y menos del 50% de ruido
(eps=2.01, min_samples=10) produce **2 clusters**: uno mayoritario con
prácticamente todos los clientes y uno minoritario formado por outliers
extremos en variables de gasto e ingresos. La silueta de 0.23 es un
**artefacto estadístico**: es alta porque los outliers están muy lejos
del macro-cluster, no porque existan segmentos realmente diferenciados.

Este patrón se repite en todas las combinaciones con <50% de ruido:
a medida que eps crece, los clusters se fusionan en uno solo y el ruido
disminuye, pero nunca emerge una segmentación con múltiples grupos
equilibrados y silueta genuina.

### ¿Por qué DBSCAN detecta densidad pero no segmentos útiles?

DBSCAN requiere que los datos presenten **regiones densas claramente
separadas por zonas de baja densidad**. Las curvas k-distance confirman
que hay dos "zonas" de densidad: la masa principal de clientes y los
outliers extremos. Pero eso no es segmentación de negocio, sino
simplemente la distinción entre clientes típicos y atípicos.

Las razones estructurales por las que no emergen segmentos útiles son:

- **Distribución continua:** las variables de gasto forman un continuo
  sin fronteras de densidad nítidas entre segmentos. No hay "vacíos"
  entre grupos de clientes medios, altos y muy altos.
- **Alta dimensionalidad:** con 16 variables numéricas, las distancias
  euclidianas se homogenizan. DBSCAN no puede distinguir regiones densas
  de escasas porque las diferencias de distancia entre puntos son pequeñas
  y uniformes en la mayoría del espacio.
- **Outliers extremos pero escasos:** los clientes de muy alto poder
  adquisitivo existen pero son demasiado pocos para formar regiones
  densas propias con cualquier min_samples razonable.

### Significado de negocio de los puntos ruido

Los 903 clientes etiquetados como ruido en la mejor combinación representan
perfiles que no se parecen suficientemente a ningún grupo denso. En términos
de negocio son **clientes atípicos o de nicho**:

- Clientes de renta muy alta con patrones de compra inusuales o poco
  frecuentes para su nivel de ingresos.
- Clientes ocasionales que no encajan en ningún segmento recurrente.
- Transiciones entre segmentos: clientes que están "entre" dos perfiles
  sin pertenecer claramente a ninguno.

Lejos de ser un problema, estos clientes podrían merecer atención
individualizada o ser excluidos de campañas masivas para no distorsionar
los mensajes dirigidos a segmentos bien definidos.

### ¿Se recomienda DBSCAN para segmentar clientes de este supermercado?

**No.** Aunque las curvas k-distance confirman que existe estructura de
densidad en los datos, esa estructura no produce segmentos de negocio
accionables: DBSCAN solo separa la masa de clientes típicos de los
outliers extremos, independientemente de los parámetros usados.
K-Means produce segmentaciones más útiles, interpretables y estables
para datos con distribución continua sin separación de densidad clara
entre segmentos.

### Dominios donde DBSCAN sí sería una excelente elección

1. **Detección de anomalías en redes o sistemas:** el tráfico de red
   legítimo forma regiones densas bien definidas; los ataques o fallos
   son puntos aislados (ruido). DBSCAN identifica los outliers sin
   necesidad de especificar cuántos tipos de anomalía existen ni asumir
   forma esférica en los clusters.

2. **Análisis geoespacial:** coordenadas GPS de entregas, accidentes o
   comercios forman clusters naturales con forma irregular (no esférica)
   separados por zonas sin datos. DBSCAN puede seguir esa geometría
   arbitraria, algo imposible para K-Means al asumir clusters esféricos
   y de tamaño similar.

In [ ]:
# =============================================================================
# CELDA 25 · Modelo de mezcla de gaussianas (GMM) con k=4
# GMM modela cada cluster como una distribución gaussiana multivariante.
# A diferencia de K-Means, la asignación es probabilística: cada punto
# recibe una probabilidad de pertenencia a cada componente.
# covariance_type="full" permite a cada gaussiana tener su propia forma,
# orientación y tamaño, siendo el tipo más flexible y adecuado cuando
# los clusters no son necesariamente esféricos.
# =============================================================================

gmm = GaussianMixture(
    n_components=K_OPTIMO,
    covariance_type="full",   # cada componente tiene matriz de covarianza propia
    random_state=RANDOM_STATE,
    n_init=5                  # múltiples inicializaciones para evitar mínimos locales
)
gmm.fit(X_scaled)

# --- Asignación de clusters al dataframe ---
df_1["Cluster_GMM"] = gmm.predict(X_scaled)

# Métricas de calidad
labels_gmm = gmm.predict(X_scaled)
sil_gmm = silhouette_score(X_scaled, labels_gmm, random_state=RANDOM_STATE)
db_gmm  = davies_bouldin_score(X_scaled, labels_gmm)

print(f"GMM — k={K_OPTIMO}, covariance_type='full'")
print(f"Silueta:        {sil_gmm:.4f}")
print(f"Davies-Bouldin: {db_gmm:.4f}")
print(f"Log-likelihood: {gmm.score(X_scaled):.4f}")
print()

# --- Probabilidades de pertenencia para los 10 primeros clientes ---
proba = gmm.predict_proba(X_scaled)   # shape (2240, 4)

df_proba = pd.DataFrame(
    proba[:10],
    columns=[f"P(Cluster {i})" for i in range(K_OPTIMO)],
    index=[f"Cliente {i}" for i in range(10)]
)
df_proba["Cluster asignado"] = gmm.predict(X_scaled[:10])

print("Probabilidades de pertenencia — primeros 10 clientes:")
display(df_proba.style
        .format({c: "{:.4f}" for c in df_proba.columns if c.startswith("P(")})
        .background_gradient(cmap="YlOrRd",
                             subset=[c for c in df_proba.columns
                                     if c.startswith("P(")]))

In [ ]:
# =============================================================================
# CELDA 26 · Identificación de cliente con asignación "blanda"
# Una asignación blanda es aquella en la que ninguna probabilidad es
# cercana a 1: el cliente tiene características compatibles con más de
# un perfil de compra simultáneamente.
# Se define "blanda" como: probabilidad máxima < 0.70
# (es decir, el cluster más probable explica menos del 70% de la pertenencia).
# =============================================================================

UMBRAL_BLANDO = 0.70   # por debajo de este valor la asignación se considera blanda

proba_max = proba.max(axis=1)   # probabilidad máxima de cada cliente

# Encontrar clientes con asignación blanda
indices_blandos = np.where(proba_max < UMBRAL_BLANDO)[0]

print(f"Clientes con asignación blanda (prob. máx < {UMBRAL_BLANDO}): "
      f"{len(indices_blandos)} de {len(proba)} "
      f"({len(indices_blandos)/len(proba)*100:.1f}%)")

if len(indices_blandos) > 0:
    # Seleccionar el cliente con la asignación más incierta (prob. máx más baja)
    idx_mas_blando = indices_blandos[proba_max[indices_blandos].argmin()]

    print(f"\nCliente más incierto: índice {idx_mas_blando}")
    print(f"Probabilidad máxima: {proba_max[idx_mas_blando]:.4f}")
    print()

    # Probabilidades de ese cliente
    df_cliente = pd.DataFrame({
        "Cluster"      : [f"Cluster {i}" for i in range(K_OPTIMO)],
        "Probabilidad" : proba[idx_mas_blando],
    }).sort_values("Probabilidad", ascending=False)

    print("Distribución de probabilidades:")
    display(df_cliente.style.format({"Probabilidad": "{:.4f}"}))

    # Perfil del cliente en variables originales
    vars_perfil = ["Age", "Income", "MntWines", "MntMeatProducts",
                   "Kidhome", "Teenhome", "NumWebPurchases",
                   "NumCatalogPurchases", "NumDealsPurchases"]
    perfil = df_1.iloc[idx_mas_blando][vars_perfil]

    print(f"\nPerfil del cliente {idx_mas_blando} (valores originales):")
    display(pd.DataFrame(perfil).T.style.format("{:.1f}"))

    # Comparación con K-Means
    cluster_gmm   = gmm.predict(X_scaled)[idx_mas_blando]
    cluster_kmeans = km_final.labels_[idx_mas_blando]

    print(f"\nAsignación GMM:    Cluster {cluster_gmm} "
          f"(prob. {proba[idx_mas_blando, cluster_gmm]:.4f})")
    print(f"Asignación K-Means: Cluster {cluster_kmeans}")
    print(f"¿Coinciden?: {'Sí' if cluster_gmm == cluster_kmeans else 'No'}")

else:
    print("No se encontraron clientes con asignación blanda "
          f"bajo el umbral de {UMBRAL_BLANDO}.")
    print("Ajustando umbral a 0.90 para análisis...")
    # Fallback: buscar con umbral más alto
    indices_blandos_alt = np.where(proba_max < 0.90)[0]
    print(f"Con umbral 0.90: {len(indices_blandos_alt)} clientes")

## GMM — Interpretación probabilística y comparación con K-Means

### Métricas de calidad

| Métrica | GMM | K-Means |
|---------|-----|---------|
| Silueta | 0.1397 | 0.1634 |
| Davies-Bouldin | 2.3863 | 1.8325 |
| Log-likelihood | -8.2748 | N/A |

GMM obtiene métricas ligeramente peores que K-Means en silueta y
Davies-Bouldin. Esto no significa que GMM sea un algoritmo inferior:
la silueta y DB miden compacidad y separación geométrica, criterios
que K-Means optimiza directamente. GMM optimiza la verosimilitud del
modelo probabilístico, un objetivo distinto. La ventaja de GMM no está
en las métricas de distancia sino en la **riqueza informativa de las
probabilidades de pertenencia**.

### Probabilidades de los 10 primeros clientes

La gran mayoría de los clientes presentan probabilidades cercanas a 0 o 1
en todos los clusters. Clientes 0, 3, 7, 8 tienen probabilidad exactamente
1.0 en un único cluster; Clientes 4 y 9 ídem. Esto indica que GMM asigna
con alta confianza a la mayoría de los puntos, coherente con la estructura
del dataset: los clusters están relativamente bien separados en las
variables de gasto e ingresos.

Las únicas asignaciones con incertidumbre visible en los 10 primeros
clientes son los índices 2, 5 y 6, que presentan probabilidades secundarias
entre el 5% y el 9% hacia el Cluster 3. Los tres están asignados al
Cluster 1 (compradores premium) pero con cierta compatibilidad con el
Cluster 3 (smart shoppers activos), dos perfiles con ingresos altos
que comparten algunas características de gasto.

### Cliente con asignación blanda — índice 2114

Se identificaron **28 clientes** con probabilidad máxima inferior a 0.70
(1.2% del total). Es un porcentaje bajo, lo que confirma que la mayoría
de los clientes tienen un perfil claro y bien diferenciado.

El cliente más incierto (índice 2114) presenta la siguiente distribución:

| Cluster | Probabilidad | Perfil asociado |
|---------|-------------|-----------------|
| Cluster 3 | 0.4951 | Smart shoppers activos |
| Cluster 2 | 0.3567 | Familias con adolescentes |
| Cluster 1 | 0.1483 | Compradores premium |
| Cluster 0 | 0.0000 | Familias jóvenes bajo presupuesto |

**Perfil del cliente 2114:**

| Variable | Valor | Centroide Cluster 3 | Centroide Cluster 2 |
|----------|-------|--------------------|--------------------|
| Age | 60 años | 49.4 | 50.4 |
| Income | 45.736 € | 59.641 € | 42.434 € |
| MntWines | 188 € | 527.5 € | 69.5 € |
| MntMeatProducts | 180 € | 147.1 € | 25.9 € |
| Kidhome | 0 | 0.2 | 0.7 |
| Teenhome | 1 | 0.9 | 1.0 |
| NumWebPurchases | 7 | 6.6 | 2.3 |
| NumCatalogPurchases | 1 | 3.3 | 0.7 |
| NumDealsPurchases | 5 | 3.7 | 2.5 |

**Interpretación en términos de comportamiento de compra:**

Este cliente tiene un perfil "puente" entre el Cluster 3 (smart shoppers
activos) y el Cluster 2 (familias con adolescentes), con un toque residual
del Cluster 1 (compradores premium).

El motivo de la incertidumbre es claro al analizar sus variables:

- **Income (45.736 €):** está entre el centroide del Cluster 2 (42.434 €)
  y el del Cluster 3 (59.641 €). No encaja plenamente en ninguno de los dos.
- **MntWines (188 €):** muy por encima del Cluster 2 (69.5 €) pero muy
  por debajo del Cluster 3 (527.5 €). Gasto intermedio sin adscripción clara.
- **MntMeatProducts (180 €):** supera claramente al Cluster 3 (147.1 €),
  acercándose al Cluster 1 (453.5 €). Explica la probabilidad residual
  del 14.8% hacia el perfil premium.
- **Teenhome (1):** compatible con ambos clusters (Cluster 2: 1.0,
  Cluster 3: 0.9). No discrimina.
- **NumWebPurchases (7):** alineado con Cluster 3 (6.6), pero
  NumCatalogPurchases (1) es más propio del Cluster 2 (0.7).

En síntesis: es un cliente mayor (60 años) con renta media-baja para su
patrón de gasto, que gasta más de lo esperado para su ingreso en carne
y vino, usa mucho la web pero poco el catálogo, y tiene un adolescente
en casa. Ningún cluster captura todos estos rasgos simultáneamente.

Desde el punto de vista de negocio, este tipo de cliente está posiblemente
en **transición de perfil**: un cliente del segmento medio que ha ido
incrementando su gasto en categorías premium sin alcanzar aún el nivel de
ingresos del segmento alto. Podría ser receptivo a campañas que combinen
descuentos (propio del Cluster 3) con productos de calidad media-alta.

### Comparación GMM vs K-Means para el cliente 2114

GMM asigna al cliente al **Cluster 3** con probabilidad **0.4951**.  
K-Means lo asigna también al **Cluster 3**.  
**Coinciden en la asignación final.**

Sin embargo, la información que aportan es radicalmente distinta:

- **K-Means** dice: *"este cliente pertenece al Cluster 3"*, sin matices.
- **GMM** dice: *"este cliente pertenece probablemente al Cluster 3
  (49.5%), pero existe una probabilidad relevante de que encaje mejor
  en el Cluster 2 (35.7%) o incluso en el Cluster 1 (14.8%)"*.

La asignación de K-Means es correcta en términos del cluster más probable,
pero oculta la incertidumbre real. En la práctica de marketing, conocer
que un cliente tiene un 36% de probabilidad de responder como el Cluster 2
es información valiosa: permite diseñar comunicaciones híbridas o testar
qué tipo de oferta responde mejor antes de asignarlo definitivamente
a una campaña segmentada.

In [ ]:
# =============================================================================
# CELDA 28 · Tabla comparativa de los cuatro algoritmos
# Todos los valores provienen de los modelos entrenados en fases anteriores.
# El porcentaje de ruido solo aplica a DBSCAN; se indica "N/A" para el resto.
# La silueta de DBSCAN se calcula sobre puntos no-ruido (eps=2.01, ms=10).
# =============================================================================

# --- Recopilar métricas ya calculadas ---
sil_kmeans = silhouette_score(X_scaled, km_final.labels_,
                               random_state=RANDOM_STATE)
db_kmeans  = davies_bouldin_score(X_scaled, km_final.labels_)

# Jerárquico Ward (ya calculado en FASE 3)
sil_ward_f = silhouette_score(X_scaled, labels_ward,
                               random_state=RANDOM_STATE)
db_ward_f  = davies_bouldin_score(X_scaled, labels_ward)

# DBSCAN — mejor combinación (eps=2.01, min_samples=10)
db_best      = DBSCAN(eps=2.01, min_samples=10)
labels_dbest = db_best.fit_predict(X_scaled)
n_ruido_best = (labels_dbest == -1).sum()
pct_ruido_best = n_ruido_best / len(labels_dbest) * 100
mask_no_ruido  = labels_dbest != -1
sil_dbscan = silhouette_score(X_scaled[mask_no_ruido],
                               labels_dbest[mask_no_ruido],
                               random_state=RANDOM_STATE)
db_dbscan  = davies_bouldin_score(X_scaled[mask_no_ruido],
                                   labels_dbest[mask_no_ruido])
n_clusters_dbscan = len(set(labels_dbest)) - 1

# GMM (ya calculado en FASE 5)
sil_gmm_f = silhouette_score(X_scaled, labels_gmm,
                              random_state=RANDOM_STATE)
db_gmm_f  = davies_bouldin_score(X_scaled, labels_gmm)

# --- Construir tabla ---
comparativa = pd.DataFrame({
    "Algoritmo": [
        "K-Means", "Jerárquico (Ward)", "DBSCAN", "GMM"
    ],
    "Silueta": [
        sil_kmeans, sil_ward_f, sil_dbscan, sil_gmm_f
    ],
    "Davies-Bouldin": [
        db_kmeans, db_ward_f, db_dbscan, db_gmm_f
    ],
    "Nº clusters": [
        K_OPTIMO, K_OPTIMO, n_clusters_dbscan, K_OPTIMO
    ],
    "% ruido": [
        "N/A", "N/A", f"{pct_ruido_best:.1f}%", "N/A"
    ],
    "Ventaja principal": [
        "Simple, escalable, centroides interpretables",
        "No requiere k previo, estructura jerárquica visible",
        "Detecta outliers explícitamente, forma arbitraria",
        "Asignación probabilística, incertidumbre cuantificada"
    ],
    "Limitación observada": [
        "Asignación dura, sensible a outliers, asume clusters esféricos",
        "Linkages complete/average colapsan con outliers extremos",
        "No adecuado para datos continuos sin estructura de densidad",
        "Peor DB, más costoso, clusters menos compactos que K-Means"
    ]
})

print("Tabla comparativa — Algoritmos de clustering")
display(comparativa.style
        .format({"Silueta": "{:.4f}", "Davies-Bouldin": "{:.4f}"})
        .highlight_max(subset=["Silueta"], color="#c6efce")
        .highlight_min(subset=["Davies-Bouldin"], color="#c6efce")
        .set_properties(**{"text-align": "left"})
        .set_caption("Comparativa de algoritmos — mismo dataset X_scaled, k=4"))

In [ ]:
# =============================================================================
# CELDA 29 · Visualización comparativa de métricas
# Dos gráficos de barras: silueta (mayor = mejor) y Davies-Bouldin (menor = mejor).
# DBSCAN se representa con asterisco en el título para recordar que su silueta
# se calcula solo sobre puntos no-ruido, no es directamente comparable.
# =============================================================================

algoritmos  = ["K-Means", "Jerárquico\n(Ward)", "DBSCAN*", "GMM"]
siluetas    = [sil_kmeans, sil_ward_f, sil_dbscan, sil_gmm_f]
db_indices  = [db_kmeans,  db_ward_f,  db_dbscan,  db_gmm_f]
colores_bar = [CLUSTER_PALETTE[i] for i in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Comparativa de métricas — Algoritmos de clustering",
             fontsize=13, fontweight="bold")

# --- Silueta ---
bars1 = axes[0].bar(algoritmos, siluetas,
                    color=colores_bar, edgecolor="white", width=0.5)
axes[0].bar_label(bars1, fmt="{:.4f}", padding=3, fontsize=10)
axes[0].set_title("Coeficiente de silueta (↑ mejor)")
axes[0].set_ylabel("Silueta")
axes[0].set_ylim(0, max(siluetas) * 1.25)
axes[0].axhline(max(siluetas), color="green", linestyle="--",
                linewidth=1, alpha=0.5, label="Mejor valor")
axes[0].legend(fontsize=9)

# --- Davies-Bouldin ---
bars2 = axes[1].bar(algoritmos, db_indices,
                    color=colores_bar, edgecolor="white", width=0.5)
axes[1].bar_label(bars2, fmt="{:.4f}", padding=3, fontsize=10)
axes[1].set_title("Índice Davies-Bouldin (↓ mejor)")
axes[1].set_ylabel("Davies-Bouldin")
axes[1].set_ylim(0, max(db_indices) * 1.25)
axes[1].axhline(min(db_indices), color="green", linestyle="--",
                linewidth=1, alpha=0.5, label="Mejor valor")
axes[1].legend(fontsize=9)

fig.text(0.5, 0.01,
         "* Silueta de DBSCAN calculada solo sobre puntos no-ruido (40.3% excluido)",
         ha="center", fontsize=9, color="gray", style="italic")

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## Comparativa final y recomendación

### Tabla resumen

| Algoritmo | Silueta | Davies-Bouldin | Nº clusters | % ruido |
|-----------|---------|----------------|-------------|---------|
| K-Means | 0.1634 | 1.8325 | 4 | N/A |
| Jerárquico (Ward) | 0.1375 | 1.9038 | 4 | N/A |
| DBSCAN* | 0.2297 | 1.0150 | 2 | 40.3% |
| GMM | 0.1397 | 2.3863 | 4 | N/A |

*Silueta y DB de DBSCAN calculadas solo sobre puntos no-ruido (59.7% del dataset).

### Advertencia sobre las métricas de DBSCAN

A primera vista DBSCAN parece el algoritmo ganador: mayor silueta (0.2297)
y menor Davies-Bouldin (1.0150). Sin embargo, estas métricas son un
**artefacto estadístico** y no deben interpretarse como superioridad real.

Hay dos sesgos acumulados que inflan artificialmente sus métricas:

1. **Exclusión del 40.3% de los datos.** La silueta y DB se calculan
   solo sobre los 1337 puntos no-ruido. Los 903 clientes etiquetados
   como ruido, que son precisamente los más difíciles de clasificar,
   quedan fuera del cálculo. Las métricas de K-Means y GMM, en cambio,
   incluyen todos los puntos.

2. **Cluster degenerado de 2 grupos.** DBSCAN produce un macro-cluster
   con prácticamente todos los clientes no-ruido y un cluster diminuto
   formado por outliers extremos. La silueta es alta porque esos outliers
   están muy lejos del macro-cluster, no porque existan segmentos
   genuinamente diferenciados. Es el mismo fenómeno observado con
   los linkages complete y average en la Tarea 3.

En términos de negocio, una segmentación de 2 clusters donde uno agrupa
al 95%+ de los clientes no es accionable para campañas de marketing
diferenciadas.

### Recomendación para uso recurrente mensual

**Algoritmo recomendado: K-Means**

Si el equipo de marketing necesita segmentar clientes de forma recurrente
cada mes, K-Means es la opción más adecuada por las siguientes razones:

**1. Interpretabilidad**
Los centroides de K-Means son directamente interpretables en unidades
originales mediante inverse_transform. El equipo de negocio puede
entender y validar cada segmento sin conocimientos estadísticos avanzados:
"el Cluster 1 gasta 583€ en vino y tiene ingresos de 76.500€" es
una descripción accionable. Los otros algoritmos no ofrecen esta
ventaja de forma tan directa.

**2. Estabilidad**
Con random_state fijo y n_init=10, K-Means produce resultados idénticos
en cada ejecución. En un proceso mensual esto es crítico: el equipo de
marketing necesita que el "Cluster 1 premium" de enero siga siendo
el mismo perfil en febrero para medir la evolución de los segmentos
y el impacto real de las campañas anteriores.

**3. Escalabilidad y coste computacional**
K-Means escala linealmente con el número de observaciones y converge
en segundos para datasets de este tamaño. GMM es más costoso
(expectation-maximization iterativo). El clustering jerárquico requiere
O(n²) en memoria y tiempo. DBSCAN necesita recalibración de eps
cada vez que cambia la distribución del dataset, lo que lo hace
poco práctico para un pipeline automatizado mensual.

**4. Clasificación de nuevos clientes sin reentrenar**
K-Means permite asignar nuevos clientes usando predict() sobre el
scaler y el modelo guardados, sin reentrenar el modelo completo.
Esto es fundamental para un pipeline mensual: los nuevos clientes
pueden clasificarse instantáneamente. El clustering jerárquico,
en cambio, no permite clasificar nuevos puntos sin reconstruir
todo el árbol.

**5. Mejor algoritmo con métricas comparables y segmentación útil**
Entre los algoritmos que producen segmentaciones accionables (4 clusters
equilibrados), K-Means obtiene la mejor silueta (0.1634) y el mejor
Davies-Bouldin (1.8325). El clustering jerárquico con Ward es comparable
pero ligeramente inferior en ambas métricas. GMM produce peor DB (2.3863)
a mayor coste computacional.

### ¿Cuándo considerar las alternativas?

- **Jerárquico (Ward):** útil para exploración inicial o cuando se
  desconoce k. El dendrograma aporta valor analítico para entender
  la estructura de los datos. No recomendado para producción recurrente
  por su coste y la imposibilidad de clasificar nuevos puntos sin
  reentrenar todo el árbol.

- **DBSCAN:** recomendable si el objetivo cambia a **detección de
  clientes anómalos** (fraude, comportamiento atípico, clientes VIP
  extremos) en lugar de segmentación masiva. En ese caso, los puntos
  de ruido son precisamente el resultado de interés.

- **GMM:** recomendable si el negocio necesita **probabilidades de
  pertenencia** para decisiones de alto coste por impacto (campañas
  caras donde equivocarse tiene coste elevado) o si se detecta que
  los clusters tienen formas elípticas muy distintas entre sí.
  La información adicional sobre incertidumbre (como el cliente 2114
  con 49.5% Cluster 3 y 35.7% Cluster 2) puede traducirse en
  estrategias de comunicación híbridas o tests A/B por segmento.